# Feed loader – CSV → Delta (append)
Python notebook (not PySpark). Loads one CSV per feed into its own Delta table in the Lakehouse, appending each run.

**Per run:** find new CSV(s) in each feed's landing folder → read as text and cast to explicit types (Int32/Int64, Decimal(18,2), Date, UTC Datetime) → add audit columns → skip if this file was already loaded → append to Delta → move CSV to `archive/`.

**Feeds are discovered**: every child folder of `LANDING_ROOT` is a feed, loaded into its own table, with column types from a `_schema.json` in the folder. Or pass one `FOLDER` (and optionally `TABLE`) to load a single location.

Within a feed, files are loaded **one at a time**, so memory depends on the largest single file, not on how many files there are. Set `PARALLEL_FEEDS` above 1 to load several feeds at once in this same session (one thread per feed); `FEED_FILTER` limits a run to named feeds. A failure in one feed stops that feed only; the others carry on, and the run fails at the end so a pipeline sees it. Files at or above `STREAM_THRESHOLD_MB` are **streamed** (`scan_csv` → `sink_delta`) and never held in memory whole; smaller files are loaded in memory, which also gives an exact row count before the write. On Polars versions without `sink_delta`, large files fall back to an in-memory load (`stream-fallback` in the summary).

Writes use row groups of up to `ROW_GROUP_ROWS` rows (8 million by default), which suits Power BI Direct Lake better than the ~1 million default. delta-rs can't apply V-Order; that needs Spark.

The engine code below is generated from `src/medallion/` in the repository (the same code the `medallion` command runs on a PC). Change it there and run `python scripts/build_notebooks.py`, rather than editing it here.

## Optional: pin library versions
Fabric's preinstalled Polars may be older than 2.0. Older versions still load everything, but files at or above `STREAM_THRESHOLD_MB` are then read whole into memory instead of streamed (the run prints which you have). To get true streaming, uncomment the line below. It must run before anything imports Polars, and it adds start-up time to every run.

In [ ]:
# %pip install -q "polars==2.0.0" "deltalake==1.6.6"

In [ ]:
# Parameters (mark this cell as a parameter cell so a pipeline can override)
WORKSPACE           = "MyWorkspace"  # Fabric workspace name
LAKEHOUSE           = "MyLakehouse"  # Lakehouse the feeds load into (bronze)
SCHEMA              = "dbo"          # set to None if the Lakehouse is not schema-enabled
DRY_RUN             = False          # True = read and check every file, write nothing
STREAM_THRESHOLD_MB = 500            # files this size or larger are streamed to Delta
LANDING_ROOT        = "landing"      # parent folder under Files; each child folder is a feed
ARCHIVE_ROOT        = "archive"      # loaded CSVs move to ARCHIVE_ROOT/<feed>
FOLDER              = ""             # "" = discover every child of LANDING_ROOT; or one folder, e.g. "landing/orders" or "incoming/acme/orders"
TABLE               = ""             # with FOLDER: target table name (overrides the schema file and the default)
TABLE_SUFFIX        = "_raw"         # default table name = folder name + suffix
MISSING_SCHEMA      = "skip"         # folder without _schema.json: "skip" (flag it), "fail", or "text" (load every column as text)
FEED_FILTER         = ""             # with discovery: only these feeds, e.g. "customers" / "orders,products"
PARALLEL_FEEDS      = 1              # feeds loaded at once in this session (1 = one after another)
ROW_GROUP_ROWS      = 8_000_000      # max rows per Parquet row group; larger groups suit Direct Lake. None = library default (~1M)
LOG_TABLE           = "load_log"     # Delta table that records every run (one row per file); None = don't log

## Engine
Shared helpers and storage access. `FabricStorage` reads and writes OneLake with the notebook's identity.

In [ ]:
# ---- Engine: src/medallion/common.py -------------------------------------------
# Generated by scripts/build_notebooks.py. Change the file in the repo, not this cell.
"""Helpers shared by the bronze, silver and gold layers."""
import inspect
import json
import uuid
from dataclasses import dataclass, field, fields
from datetime import datetime, timezone

import polars as pl
import deltalake
from deltalake import DeltaTable

TS = pl.Datetime("us", "UTC")


def utcnow():
    return datetime.now(timezone.utc)


def new_run_id():
    return str(uuid.uuid4())


def first_line(err, limit=1000):
    """The first line of an exception message, shortened for logs and summaries."""
    return str(err).splitlines()[0][:limit] if str(err) else type(err).__name__


def collect_streaming(lf):
    """Run a lazy query with the streaming engine on any Polars version."""
    try:
        return lf.collect(engine="streaming")     # Polars 1.2x and later
    except (TypeError, ValueError):
        return lf.collect(streaming=True)         # older Polars


def schema_of(lf):
    """A LazyFrame's column names and types, on any Polars version."""
    try:
        return dict(lf.collect_schema())
    except AttributeError:
        return dict(lf.schema)                    # older Polars


def write_options(row_group_rows):
    """delta-rs write options: new source columns allowed, large row groups for Direct Lake."""
    opts = {"schema_mode": "merge"}
    if row_group_rows:
        from deltalake import WriterProperties
        opts["writer_properties"] = WriterProperties(max_row_group_size=row_group_rows)
    return opts


def merge_kwargs(dt, predicate, write_opts):
    """Arguments for DeltaTable.merge: streamed execution where available (less memory)."""
    params = inspect.signature(dt.merge).parameters
    kwargs = {"predicate": predicate, "source_alias": "s", "target_alias": "t"}
    if "streamed_exec" in params:
        kwargs["streamed_exec"] = True
    if "writer_properties" in params and "writer_properties" in write_opts:
        kwargs["writer_properties"] = write_opts["writer_properties"]
    return kwargs


def compact(path, storage, log=print, retention_hours=168):
    """Compact small files and vacuum old ones in one Delta table (if it exists)."""
    if not DeltaTable.is_deltatable(path, storage_options=storage.options()):
        return
    dt = DeltaTable(path, storage_options=storage.options())
    log(f"{path.rsplit('/', 1)[-1]}: {dt.optimize.compact()}")
    dt.vacuum(retention_hours=retention_hours, dry_run=False, enforce_retention_duration=True)


# --- Parameters ---------------------------------------------------------------
def param(default, help=""):
    """A run parameter: its default, and the comment shown in the notebook's parameter cell."""
    return field(default=default, metadata={"help": help})


class ParamsBase:
    """Base for each layer's parameters. Field names match the notebook's parameter cell."""

    @classmethod
    def from_mapping(cls, values):
        """Build from a dict (or a notebook's globals()), taking only this layer's parameters."""
        names = {f.name for f in fields(cls)}
        return cls(**{k: v for k, v in values.items() if k in names})

    def updated(self, **changes):
        """A copy with some parameters changed. Unknown names are an error."""
        names = {f.name for f in fields(self)}
        unknown = sorted(set(changes) - names)
        if unknown:
            raise ValueError(f"unknown parameter(s) for {type(self).__name__}: {unknown}")
        values = {f.name: getattr(self, f.name) for f in fields(self)}
        values.update(changes)
        return type(self)(**values)


# --- Results ------------------------------------------------------------------
@dataclass
class RunResult:
    """What a layer run returns: the summary table, the notebook exit value, and any problems."""
    layer: str
    run_id: str
    summary: pl.DataFrame
    exit_value: dict
    problems: list = field(default_factory=list)

    @property
    def ok(self):
        return not self.problems

    def exit_json(self):
        return json.dumps(self.exit_value)

    def check(self):
        """Raise if anything failed, so a pipeline (or the CLI) marks the run failed."""
        if self.problems:
            raise RuntimeError(f"{len(self.problems)} problem(s): " + "; ".join(self.problems))


def versions():
    out = {"polars": pl.__version__, "deltalake": deltalake.__version__}
    try:
        import duckdb
        out["duckdb"] = duckdb.__version__
    except ImportError:
        pass
    return out


In [ ]:
# ---- Engine: src/medallion/storage.py -------------------------------------------
# Generated by scripts/build_notebooks.py. Change the file in the repo, not this cell.
"""Where the Lakehouses live.

Every layer reads and writes through a storage object, so the same code runs:
- in a Fabric notebook, against OneLake (FabricStorage, using notebookutils);
- on a PC, against local folders laid out like Lakehouses (LocalStorage).

A storage object gives each Lakehouse a base path (an abfss:// URL or a local folder), the
storage_options delta-rs and Polars need for that path, and the few file operations the
layers use: list a folder, read a small text file, move a file.
"""
import os
import shutil
from dataclasses import dataclass
from pathlib import Path


@dataclass
class Entry:
    """One item in a folder listing."""
    name: str
    path: str
    is_dir: bool
    size: int


def _enc(name):
    return name.replace(" ", "%20")   # spaces in workspace/lakehouse names must be encoded


def tables_root(base, schema):
    return f"{base}/Tables/{schema}" if schema else f"{base}/Tables"


class FabricStorage:
    """OneLake, from inside a Fabric notebook (authenticates with notebookutils)."""
    kind = "fabric"

    def __init__(self):
        import notebookutils
        self._nu = notebookutils

    def lakehouse(self, workspace, lakehouse):
        return f"abfss://{_enc(workspace)}@onelake.dfs.fabric.microsoft.com/{_enc(lakehouse)}.Lakehouse"

    def options(self):
        # fetched per call so long runs don't hit token expiry
        return {"bearer_token": self._nu.credentials.getToken("storage"),
                "use_fabric_endpoint": "true"}

    def ls(self, path):
        return [Entry(f.name, f.path, f.isDir, 0 if f.isDir else f.size) for f in self._nu.fs.ls(path)]

    def read_text(self, path, max_bytes=10_000_000):
        return self._nu.fs.head(path, max_bytes)

    def move(self, src, dest):
        self._nu.fs.mv(src, dest, True, True)    # create_path, overwrite

    def context(self):
        """Notebook and pipeline details from Fabric."""
        try:
            return dict(self._nu.runtime.context)
        except Exception:
            return {}


class LocalStorage:
    """Local folders: <root>/<lakehouse>.Lakehouse/{Files,Tables}. The workspace name is ignored."""
    kind = "local"

    def __init__(self, root):
        self.root = Path(root).expanduser().resolve()

    def lakehouse(self, workspace, lakehouse):
        return (self.root / f"{lakehouse}.Lakehouse").as_posix()

    def options(self):
        return None

    def ls(self, path):
        names = sorted(os.listdir(path))           # raises if the folder doesn't exist, like OneLake
        out = []
        for n in names:
            p = os.path.join(path, n)
            d = os.path.isdir(p)
            out.append(Entry(n, Path(p).as_posix(), d, 0 if d else os.path.getsize(p)))
        return out

    def read_text(self, path, max_bytes=10_000_000):
        with open(path, encoding="utf-8") as fh:
            return fh.read(max_bytes)

    def move(self, src, dest):
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if os.path.exists(dest):
            os.remove(dest)
        shutil.move(src, dest)

    def context(self):
        return {"currentNotebookName": "medallion (local)", "isForPipeline": False,
                "userName": os.environ.get("USERNAME") or os.environ.get("USER")}


## Feed discovery and schema files
Each child folder of `LANDING_ROOT` is a feed. Its table is named after the folder (`orders` → `orders_raw`) and loaded CSVs move to `ARCHIVE_ROOT/<folder>`. Column types come from a `_schema.json` file in the folder:

```json
{
  "table": "orders_raw",
  "columns": {"order_id": "Int64", "customer_id": "Int32", "order_date": "Date", "amount": "Decimal(18,2)"},
  "formats": {"order_date": "%Y-%m-%d"},
  "csv_options": {"separator": ",", "has_header": true}
}
```

`columns` lists every CSV column in file order. Types: `String`, `Int8`–`Int64`, `UInt8`–`UInt64`, `Float32`, `Float64`, `Boolean`, `Date`, `Datetime` (stored as UTC, microseconds) and `Decimal(p,s)`. `Date` and `Datetime` columns need a format in `formats`. `table` and `csv_options` are optional.

In [ ]:
# ---- Engine: src/medallion/bronze.py -------------------------------------------
# Generated by scripts/build_notebooks.py. Change the file in the repo, not this cell.
"""Bronze: landing CSVs -> raw Delta tables (append), typed by a _schema.json per folder."""
import gc
import json
import re
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

import polars as pl
from deltalake import DeltaTable



@dataclass
class BronzeParams(ParamsBase):
    WORKSPACE: str = param("MyWorkspace", "Fabric workspace name")
    LAKEHOUSE: str = param("MyLakehouse", "Lakehouse the feeds load into (bronze)")
    SCHEMA: str = param("dbo", "set to None if the Lakehouse is not schema-enabled")
    DRY_RUN: bool = param(False, "True = read and check every file, write nothing")
    STREAM_THRESHOLD_MB: int = param(500, "files this size or larger are streamed to Delta")
    LANDING_ROOT: str = param("landing", "parent folder under Files; each child folder is a feed")
    ARCHIVE_ROOT: str = param("archive", "loaded CSVs move to ARCHIVE_ROOT/<feed>")
    FOLDER: str = param("", '"" = discover every child of LANDING_ROOT; or one folder, e.g. "landing/orders" or "incoming/acme/orders"')
    TABLE: str = param("", "with FOLDER: target table name (overrides the schema file and the default)")
    TABLE_SUFFIX: str = param("_raw", "default table name = folder name + suffix")
    MISSING_SCHEMA: str = param("skip", 'folder without _schema.json: "skip" (flag it), "fail", or "text" (load every column as text)')
    FEED_FILTER: str = param("", 'with discovery: only these feeds, e.g. "customers" / "orders,products"')
    PARALLEL_FEEDS: int = param(1, "feeds loaded at once in this session (1 = one after another)")
    ROW_GROUP_ROWS: int = param(8_000_000, "max rows per Parquet row group; larger groups suit Direct Lake. None = library default (~1M)")
    LOG_TABLE: str = param("load_log", "Delta table that records every run (one row per file); None = don't log")


# --- Version check: which features this Polars has ---------------------------
HAS_SINK_DELTA = hasattr(pl.LazyFrame, "sink_delta")   # true streaming to Delta

# --- Schema files ---------------------------------------------------------------
TYPE_MAP = {
    "string": pl.Utf8, "utf8": pl.Utf8, "boolean": pl.Boolean, "bool": pl.Boolean,
    "int8": pl.Int8, "int16": pl.Int16, "int32": pl.Int32, "int64": pl.Int64,
    "uint8": pl.UInt8, "uint16": pl.UInt16, "uint32": pl.UInt32, "uint64": pl.UInt64,
    "float32": pl.Float32, "float64": pl.Float64, "date": pl.Date,
    "datetime": pl.Datetime("us", "UTC"),
}

DEFAULT_CSV_OPTIONS = {"separator": ",", "has_header": True}


def parse_type(name, where):
    key = name.strip().lower().replace(" ", "")
    m = re.fullmatch(r"decimal\((\d+),(\d+)\)", key)
    if m:
        return pl.Decimal(int(m.group(1)), int(m.group(2)))
    if key in TYPE_MAP:
        return TYPE_MAP[key]
    raise ValueError(f"{where}: unknown type '{name}'")


def safe_name(folder):
    """Folder name -> table/feed name: lower case, letters, digits and underscores."""
    return re.sub(r"[^a-z0-9_]+", "_", folder.strip().lower()).strip("_")


def parse_schema(text, where):
    """A _schema.json file's text -> columns (Polars types), formats and CSV options."""
    spec = json.loads(text)
    if not spec.get("columns"):
        raise ValueError(f"{where}: 'columns' is missing or empty")
    columns = {c: parse_type(t, f"{where} column '{c}'") for c, t in spec["columns"].items()}
    formats = spec.get("formats", {})
    for c, t in columns.items():
        if (t == pl.Date or isinstance(t, pl.Datetime)) and c not in formats:
            raise ValueError(f"{where}: column '{c}' is {t} but has no entry in 'formats'")
    return {"table": spec.get("table"), "columns": columns, "formats": formats,
            "csv_options": spec.get("csv_options", dict(DEFAULT_CSV_OPTIONS))}


def typed_select(lf, cfg):
    """Cast text columns strictly to the schema's types, so a bad value fails the load
    instead of changing a column type."""
    exprs = []
    for c, dtype in cfg["columns"].items():
        e = pl.col(c).str.strip_chars()
        if dtype == pl.Date:
            e = e.str.to_date(cfg["formats"][c], strict=True)
        elif isinstance(dtype, pl.Datetime):
            e = e.str.to_datetime(cfg["formats"][c], time_unit="us", strict=True)
            if dtype.time_zone:
                e = e.dt.replace_time_zone(dtype.time_zone)
        elif dtype == pl.Boolean:
            e = e.str.to_lowercase().replace_strict(
                {"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False},
                return_dtype=pl.Boolean)
        elif dtype != pl.Utf8:
            e = e.cast(dtype, strict=True)
        exprs.append(e.alias(c))
    return lf.select(exprs)


def add_audit(lf, file_name, load_ts):
    return lf.with_columns(
        pl.lit(file_name).alias("_source_file"),
        pl.lit(load_ts).cast(TS).alias("_load_ts"),
    )


SUMMARY_SCHEMA = {
    "feed": pl.Utf8, "folder": pl.Utf8, "target_table": pl.Utf8, "file": pl.Utf8,
    "size_mb": pl.Float64, "mode": pl.Utf8, "status": pl.Utf8, "rows": pl.Int64,
    "started_utc": TS, "finished_utc": TS, "seconds": pl.Float64, "error": pl.Utf8}


def result_row(feed, cfg, file="", size_mb=None, mode="", status="", rows=0,
               started=None, finished=None, error=None):
    return {"feed": feed, "folder": cfg["landing"], "target_table": cfg["table"], "file": file,
            "size_mb": None if size_mb is None else round(size_mb, 1), "mode": mode,
            "status": status, "rows": rows, "started_utc": started, "finished_utc": finished,
            "seconds": round((finished - started).total_seconds(), 1) if started and finished else 0.0,
            "error": error}


class BronzeRun:
    """One run of the feed loader."""

    def __init__(self, params, storage, log=print):
        self.p, self.storage, self.log = params, storage, log
        self.base = storage.lakehouse(params.WORKSPACE, params.LAKEHOUSE)
        self.write_opts = write_options(params.ROW_GROUP_ROWS)
        self.run_id = new_run_id()
        self.load_ts = utcnow()

    # --- paths ----------------------------------------------------------------
    def table_path(self, name):
        return f"{tables_root(self.base, self.p.SCHEMA)}/{name}"

    def files_path(self, rel):
        return f"{self.base}/Files/{rel}"

    # --- feed discovery ----------------------------------------------------------
    def read_schema_file(self, rel_folder):
        """Parsed _schema.json for a folder, or None if it has none."""
        try:
            text = self.storage.read_text(self.files_path(f"{rel_folder}/_schema.json"), 1_000_000)
        except Exception:
            return None
        return parse_schema(text, f"{rel_folder}/_schema.json")

    def build_feed(self, rel_folder, table_override=""):
        p = self.p
        name = safe_name(rel_folder.rstrip("/").rsplit("/", 1)[-1])
        spec = self.read_schema_file(rel_folder)
        if spec is None and p.MISSING_SCHEMA == "text":
            spec = {"table": None, "columns": None, "formats": {}, "csv_options": dict(DEFAULT_CSV_OPTIONS)}
        # archive mirrors the landing path: landing/orders -> archive/orders, incoming/acme/orders -> archive/incoming/acme/orders
        root = p.LANDING_ROOT
        rel_under_root = rel_folder[len(root) + 1:] if rel_folder.startswith(root + "/") else rel_folder
        feed = {"landing": rel_folder, "archive": f"{p.ARCHIVE_ROOT}/{rel_under_root}",
                "table": table_override or (spec or {}).get("table") or f"{name}{p.TABLE_SUFFIX}",
                "schema_source": "_schema.json" if spec and spec["columns"] else
                                 ("all text" if spec else "missing")}
        if spec:
            feed.update(columns=spec["columns"], formats=spec["formats"], csv_options=spec["csv_options"])
        return name, feed

    def subfolders(self, rel_folder):
        try:
            return sorted(f.name for f in self.storage.ls(self.files_path(rel_folder)) if f.is_dir)
        except Exception:
            return []

    def discover(self):
        """The feeds for this run, and result rows for folders skipped for having no schema file."""
        p = self.p
        if p.FOLDER:
            feeds = dict([self.build_feed(p.FOLDER.strip("/"), p.TABLE)])
        else:
            if p.TABLE:
                raise ValueError("TABLE can only be used together with FOLDER")
            feeds = dict(self.build_feed(f"{p.LANDING_ROOT}/{d}") for d in self.subfolders(p.LANDING_ROOT))
            if p.FEED_FILTER:
                wanted = {safe_name(f) for f in p.FEED_FILTER.split(",") if f.strip()}
                unknown = wanted - feeds.keys()
                if unknown:
                    raise ValueError(f"FEED_FILTER names feeds with no folder under {p.LANDING_ROOT}: {sorted(unknown)}")
                feeds = {k: v for k, v in feeds.items() if k in wanted}

        # two folders must never load into the same table
        tables = {}
        for name, f in feeds.items():
            if f["table"] in tables:
                raise ValueError(f"Folders '{tables[f['table']]}' and '{f['landing']}' both map to table {f['table']}")
            tables[f["table"]] = f["landing"]

        # folders without a schema file
        missing = {k: v for k, v in feeds.items() if v["schema_source"] == "missing"}
        if missing and p.MISSING_SCHEMA == "fail":
            raise ValueError("No _schema.json in: " + ", ".join(v["landing"] for v in missing.values()))
        skipped = [{"feed": k, "folder": v["landing"], "target_table": v["table"], "file": "", "size_mb": None,
                    "mode": "", "status": "no schema", "rows": 0, "started_utc": None, "finished_utc": None,
                    "seconds": 0.0, "error": f"no _schema.json in {v['landing']}; files left in place"}
                   for k, v in missing.items()]
        feeds = {k: v for k, v in feeds.items() if k not in missing}

        for k, v in feeds.items():
            self.log(f"{k:<20} {v['landing']:<35} -> {v['table']:<25} ({v['schema_source']})")
        for r in skipped:
            self.log(f"{r['feed']:<20} SKIPPED: {r['error']}")
        if not feeds:
            self.log("No feeds to load.")
        return feeds, skipped

    # --- loading -----------------------------------------------------------------
    def list_csvs(self, rel_folder):
        try:
            items = self.storage.ls(self.files_path(rel_folder))
        except Exception:
            return []
        # (path, size in MB), sorted by name
        return sorted((f.path, f.size / 1024**2) for f in items if f.name.lower().endswith(".csv"))

    def rows_for_file(self, target, file_name):
        """Rows in the table from this source file (0 if the table doesn't exist yet).
        Fast even on large tables: every row from one file has the same _source_file,
        so delta-rs skips other files' Parquet using their min/max stats."""
        if not DeltaTable.is_deltatable(target, storage_options=self.storage.options()):
            return 0
        return (pl.scan_delta(target, storage_options=self.storage.options())
                  .filter(pl.col("_source_file") == file_name)
                  .select(pl.len()).collect().item())

    def already_loaded(self, target, file_name):
        """Idempotency guard: has this source file already been appended?"""
        return self.rows_for_file(target, file_name) > 0

    def scan_feed_csv(self, path, cfg):
        """Lazy, typed read of one CSV. Everything is read as text, then cast strictly."""
        if cfg["columns"] is None:      # MISSING_SCHEMA = "text": the file's own header, every column as text
            return pl.scan_csv(path, infer_schema=False, storage_options=self.storage.options(),
                               null_values=["", "NULL", "null"], **cfg["csv_options"])
        lf = pl.scan_csv(path, schema={c: pl.Utf8 for c in cfg["columns"]},
                         storage_options=self.storage.options(),
                         null_values=["", "NULL", "null"], **cfg["csv_options"])
        return typed_select(lf, cfg)

    def load_feed(self, feed, cfg):
        """Load every new CSV for one feed, one file at a time.
        Returns one result row per file. A failure stops this feed only:
        the failed file stays in landing and later files wait for the next run."""
        target = self.table_path(cfg["table"])
        files = self.list_csvs(cfg["landing"])
        if not files:
            self.log(f"[{feed}] no new CSVs in {cfg['landing']}")
            now = utcnow()
            return [result_row(feed, cfg, status="no new files", started=now, finished=now)]
        out = []
        for src, size_mb in files:
            file_name = src.rsplit("/", 1)[-1]
            stream = size_mb >= self.p.STREAM_THRESHOLD_MB
            mode = ("stream" if HAS_SINK_DELTA else "stream-fallback") if stream else "memory"
            started = utcnow()
            try:
                status, rows = self.load_file(feed, cfg, target, src, file_name, size_mb, stream, mode)
            except Exception as e:
                msg = first_line(e)
                self.log(f"[{feed}] {file_name} FAILED: {msg}")
                out.append(result_row(feed, cfg, file_name, size_mb, mode, "failed", 0, started, utcnow(), msg))
                break
            out.append(result_row(feed, cfg, file_name, size_mb, mode, status, rows, started, utcnow()))
        return out

    def load_file(self, feed, cfg, target, src, file_name, size_mb, stream, mode):
        dry_run, opts = self.p.DRY_RUN, self.storage.options
        if self.already_loaded(target, file_name):
            self.log(f"[{feed}] {file_name} already loaded – archiving only")
            status, rows = "skipped", 0
        else:
            lf = add_audit(self.scan_feed_csv(src, cfg), file_name, self.load_ts)
            if dry_run:
                # null_count forces every cast to run, so bad values still fail here
                rows = collect_streaming(lf.select([pl.len().alias("_rows"), pl.all().null_count()]))["_rows"][0]
                status = "dry run"
            elif stream and HAS_SINK_DELTA:
                # never holds the whole file in memory; one Delta commit at the end
                lf.sink_delta(target, mode="append", storage_options=opts(), delta_write_options=self.write_opts)
                rows = self.rows_for_file(target, file_name)
                status = "loaded"
            else:
                # in memory; large files on older Polars also come here (streaming engine for the read)
                df = collect_streaming(lf) if stream else lf.collect()
                rows = df.height
                df.write_delta(target, mode="append", storage_options=opts(), delta_write_options=self.write_opts)
                del df                      # free this file before reading the next
                status = "loaded"
            del lf
            gc.collect()
            verb = "checked" if dry_run else "appended"
            self.log(f"[{feed}] {file_name} ({size_mb:,.0f} MB, {mode}): {verb} {rows:,} rows -> {cfg['table']}")
        if not dry_run:
            self.storage.move(src, self.files_path(f"{cfg['archive']}/{file_name}"))
        return status, rows

    # --- log -----------------------------------------------------------------------
    def write_log(self, summary, started, finished, workers):
        """Append this run's rows to LOG_TABLE. Returns an error message, or None."""
        p = self.p
        ctx = self.storage.context()
        pick = lambda *keys: next((str(ctx[k]) for k in keys if ctx.get(k) not in (None, "")), None)
        v = versions()
        log_df = summary.with_columns(
            pl.lit(self.run_id).alias("run_id"),
            pl.lit(started).cast(TS).alias("run_started_utc"),
            pl.lit(finished).cast(TS).alias("run_finished_utc"),
            pl.lit(p.WORKSPACE).alias("workspace"),
            pl.lit(p.LAKEHOUSE).alias("lakehouse"),
            pl.lit(pick("currentNotebookName"), dtype=pl.Utf8).alias("notebook_name"),
            pl.lit(str(ctx.get("isForPipeline", "")).lower() == "true").alias("triggered_by_pipeline"),
            pl.lit(pick("pipelineJobId", "pipelineRunId"), dtype=pl.Utf8).alias("pipeline_run_id"),
            pl.lit(pick("userName"), dtype=pl.Utf8).alias("user_name"),
            pl.lit(p.FOLDER or p.LANDING_ROOT).alias("source_root"),
            pl.lit(workers).cast(pl.Int32).alias("parallel_feeds"),
            pl.lit(v["polars"]).alias("polars_version"),
            pl.lit(v["deltalake"]).alias("deltalake_version"),
            pl.lit(json.dumps(ctx, default=str) if ctx else None, dtype=pl.Utf8).alias("context_json"),
        ).select(
            "run_id", "run_started_utc", "run_finished_utc", "workspace", "lakehouse",
            "feed", "folder", "target_table", "file", "size_mb", "mode", "status", "rows",
            "started_utc", "finished_utc", "seconds", "error",
            "notebook_name", "triggered_by_pipeline", "pipeline_run_id", "user_name",
            "source_root", "parallel_feeds", "polars_version", "deltalake_version", "context_json",
        )
        try:
            log_df.write_delta(self.table_path(p.LOG_TABLE), mode="append",
                               storage_options=self.storage.options(),
                               delta_write_options={"schema_mode": "merge"})
            self.log(f"Logged {log_df.height} row(s) to {p.LOG_TABLE} (run_id {self.run_id})")
            return None
        except Exception as e:
            err = first_line(e, 500)
            self.log(f"WARNING: could not write the load log: {err}")
            return err

    # --- run -------------------------------------------------------------------------
    def run(self):
        v = versions()
        self.log(f"polars {v['polars']}, deltalake {v['deltalake']} ({self.storage.kind} storage)")
        if not HAS_SINK_DELTA:
            self.log("NOTE: this Polars has no sink_delta, so large files are read whole into memory "
                     "instead of streamed. Pin a newer version for true streaming.")
        feeds, skipped = self.discover()

        # Run the feeds: one after another, or several at once on threads.
        # Polars and delta-rs release Python's GIL while they work, so threads run in parallel.
        started = utcnow()
        workers = max(1, min(self.p.PARALLEL_FEEDS, len(feeds) or 1))
        if workers == 1:
            per_feed = [self.load_feed(feed, cfg) for feed, cfg in feeds.items()]
        else:
            with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="feed") as pool:
                per_feed = list(pool.map(lambda kv: self.load_feed(*kv), feeds.items()))
        results = [r for feed_rows in per_feed for r in feed_rows] + skipped
        finished = utcnow()
        self.log(f"Run finished in {(finished - started).total_seconds():,.0f}s with {workers} feed(s) at a time")
        summary = pl.DataFrame(results, schema=SUMMARY_SCHEMA)

        # The log is written even when a feed failed, so failures are recorded too. Dry runs aren't logged.
        log_error = None
        if self.p.LOG_TABLE and not self.p.DRY_RUN:
            log_error = self.write_log(summary, started, finished, workers)
        elif self.p.DRY_RUN:
            self.log("Dry run: load log not written")

        problems = [f"{r['feed']}/{r['file']}: {r['error']}"
                    for r in summary.filter(pl.col("status") == "failed").iter_rows(named=True)]
        if log_error:
            problems.append(f"load log: {log_error}")
        exit_value = {"run_id": self.run_id,
                      "files": json.loads(summary.drop("started_utc", "finished_utc").write_json())}
        self.feeds = feeds
        return RunResult("bronze", self.run_id, summary, exit_value, problems)

    def maintain(self):
        """Compact and vacuum every feed's table. Appends add files every run."""
        feeds, _ = self.discover()
        for cfg in feeds.values():
            compact(self.table_path(cfg["table"]), self.storage, self.log)


def run(params, storage, log=print):
    """Load every new landing CSV. Returns a RunResult."""
    return BronzeRun(params, storage, log).run()


def maintain(params, storage, log=print):
    BronzeRun(params, storage, log).maintain()


## Load
Loads the feeds, then appends the results to the `LOG_TABLE` Delta table: one row per file (plus one per feed with no new files or no schema), with the run's ID, start and end times, where it ran from, and the library versions. Dry runs aren't logged. The log is written even when a feed failed, so failures are recorded too.

In [ ]:
params = BronzeParams.from_mapping(globals())
result = BronzeRun(params, FabricStorage()).run()
result.summary.select("feed", "file", "mode", "status", "rows", "started_utc", "seconds", "error")

## Maintenance (optional)
Appends add files every run. Compact periodically – e.g. run this cell weekly or after a large batch, or move it to its own scheduled notebook.

In [ ]:
RUN_MAINTENANCE = False

if RUN_MAINTENANCE:
    BronzeRun(params, FabricStorage()).maintain()

In [ ]:
# Fail the run if any feed (or the log write) failed, so a pipeline marks it failed;
# otherwise return the summary with the run ID
import notebookutils
result.check()
notebookutils.notebook.exit(result.exit_json())